# REInt AI: Statistical Analysis for Forecast Accuracy
This notebook analyzes wind generation forecasts (WINDFOR) vs actual outputs (FUELHH) for the month of January 2025.

**Metrics:**
1. **MAE (Mean Absolute Error)** for multiple forecast horizons.
2. **P95 Reliability Floor**: The generation level that is exceeded 95% of the time.
3. **Error Distribution**: Analyzing bias (over/under forecasting).
4. **Time of Day Heatmap**: Diurnal patterns of forecast errors.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

# Set up visual aesthetics
plt.style.use('dark_background')
sns.set_palette(['#39ff14', '#0088ff', '#ff39a8', '#fce803', '#ffffff'])
sns.set_style("darkgrid", {"axes.facecolor": "#111111", "figure.facecolor": "#0a0a0a"})

# Load the data generated by fetch_january.py
actuals = pd.read_csv('data/jan2025_actuals.csv')
forecasts = pd.read_csv('data/jan2025_forecasts.csv')

actuals['startTime'] = pd.to_datetime(actuals['startTime'])
forecasts['startTime'] = pd.to_datetime(forecasts['startTime'])
forecasts['publishTime'] = pd.to_datetime(forecasts['publishTime'])

print(f"Loaded {len(actuals)} actual measurements and {len(forecasts)} forecast data points.")


## 1. P95 Reliability Floor
Calculate the generation level that is exceeded 95% of the time in January 2025. (This represents the 5th percentile of actual generation).


In [ ]:
# Drop NaNs just in case
valid_actuals = actuals.dropna(subset=['actual_generation'])

# 5th percentile = exceeded 95% of the time
p95_floor = np.percentile(valid_actuals['actual_generation'], 5)

print(f"P95 Reliability Floor for January 2025: {p95_floor:.2f} MW")

plt.figure(figsize=(14, 5))
plt.plot(valid_actuals['startTime'], valid_actuals['actual_generation'], color='#0088ff', label='Actual Wind Generation')
plt.axhline(y=p95_floor, color='#ff39a8', linestyle='--', linewidth=2, label=f'P95 Floor ({p95_floor:.2f} MW)')
plt.title('Wind Generation w/ P95 Reliability Floor')
plt.xlabel('Date')
plt.ylabel('Generation (MW)')
plt.legend()
plt.tight_layout()
plt.show()


## 2. MAE (Mean Absolute Error) by Horizon
Calculate MAE for horizons: 1h, 4h, 12h, 24h, and 48h.


In [ ]:
horizons = [1, 4, 12, 24, 48]
mae_results = {}

# Calculate lead times in hours
forecasts['lead_time_hours'] = (forecasts['startTime'] - forecasts['publishTime']).dt.total_seconds() / 3600

for h in horizons:
    # Filter for valid horizon: publishTime <= startTime - h -> lead_time >= h
    filtered = forecasts[forecasts['lead_time_hours'] >= h]
    
    # Get latest publish time for each target time
    idx = filtered.groupby('startTime')['publishTime'].idxmax()
    best_fcst = filtered.loc[idx]
    
    # Merge with actuals
    merged = pd.merge(actuals, best_fcst[['startTime', 'forecasted_generation']], on='startTime', how='inner')
    merged.dropna(inplace=True)
    
    # Calculate MAE
    if not merged.empty:
        mae = np.mean(np.abs(merged['forecasted_generation'] - merged['actual_generation']))
        mae_results[f"{h}h"] = mae
    else:
        mae_results[f"{h}h"] = None

print("Mean Absolute Error (MAE) by Horizon:")
for k, v in mae_results.items():
    print(f"Horizon {k}: {v:.2f} MW" if v is not None else f"Horizon {k}: No Data")

# Bar Chart
plt.figure(figsize=(8, 5))
x = list(mae_results.keys())
y = [v for v in mae_results.values()]
sns.barplot(x=x, y=y, color='#39ff14')
plt.title('Mean Absolute Error (MAE) vs Forecast Horizon')
plt.xlabel('Forecast Horizon')
plt.ylabel('MAE (MW)')
for i, v in enumerate(y):
    if v is not None:
        plt.text(i, v + 50, f"{v:.0f}", ha='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Error Distribution (Bias Analysis)
Calculate Error = Forecasted - Actual. A positive error means the model over-forecasted. Let's look at the 24h horizon.


In [ ]:
# Get 24h horizon dataset
filtered_24 = forecasts[forecasts['lead_time_hours'] >= 24]
idx_24 = filtered_24.groupby('startTime')['publishTime'].idxmax()
best_fcst_24 = filtered_24.loc[idx_24]
merged_24 = pd.merge(actuals, best_fcst_24[['startTime', 'forecasted_generation']], on='startTime', how='inner').dropna()

if not merged_24.empty:
    merged_24['error'] = merged_24['forecasted_generation'] - merged_24['actual_generation']

    plt.figure(figsize=(10, 6))
    sns.histplot(merged_24['error'], bins=50, kde=True, color='#fce803')
    plt.axvline(x=0, color='white', linestyle='--', linewidth=1.5)
    plt.axvline(x=merged_24['error'].mean(), color='#ff39a8', linestyle='-', linewidth=2, label=f"Mean Error (Bias): {merged_24['error'].mean():.2f} MW")
    plt.title('Error Distribution (24h Forecast Horizon)')
    plt.xlabel('Error (MW) [Positive = Over-forecast]')
    plt.ylabel('Frequency')
    plt.legend()
    plt.tight_layout()
    plt.show()

    bias = merged_24['error'].mean()
    if bias > 0:
        print(f"The model has a POSITIVE bias, tending to OVER-forecast by {bias:.2f} MW on average at 24h.")
    else:
        print(f"The model has a NEGATIVE bias, tending to UNDER-forecast by {abs(bias):.2f} MW on average at 24h.")
else:
    print("Not enough 24h horizon data to plot error distribution.")


## 4. Diurnal Error Heatmap
Visualize how the magnitude of the error (Absolute Error) changes throughout the 24 hours of the day (diurnal patterns).


In [ ]:
if not merged_24.empty:
    # Calculate absolute error
    merged_24['abs_error'] = np.abs(merged_24['error'])

    # Extract Hour and Day
    merged_24['hour'] = merged_24['startTime'].dt.hour
    merged_24['day'] = merged_24['startTime'].dt.day

    # Create a pivot table for the heatmap
    heatmap_data = merged_24.pivot_table(values='abs_error', index='hour', columns='day', aggfunc='mean')

    plt.figure(figsize=(14, 8))
    sns.heatmap(heatmap_data, cmap='viridis', cbar_kws={'label': 'Absolute Error (MW)'})
    plt.title('Diurnal Pattern: Absolute Error vs Time of Day (24h Horizon)')
    plt.xlabel('Day in January 2025')
    plt.ylabel('Hour of Day (UTC)')
    plt.tight_layout()
    plt.show()

    # Let's also do a simple line chart summarizing average error by hour of day
    hourly_mean = merged_24.groupby('hour')['abs_error'].mean().reset_index()

    plt.figure(figsize=(10, 5))
    sns.lineplot(data=hourly_mean, x='hour', y='abs_error', color='#39ff14', marker='o')
    plt.title('Average Absolute Error by Hour of Day')
    plt.xlabel('Hour of Day (UTC)')
    plt.ylabel('Mean Absolute Error (MW)')
    plt.xticks(range(0, 24))
    plt.tight_layout()
    plt.show()
else:
    print("Not enough 24h horizon data for diurnal heatmap.")
